# **Setup**

In [ ]:
!pip install pytorch_lightning wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl
import wandb

from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.loggers import WandbLogger

from sklearn.model_selection import train_test_split 
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error, mean_absolute_error

import kagglehub
from dotenv import load_dotenv

def set_seed(seed: int = 67):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 1488
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

#### **Dataset**

In [3]:
load_dotenv() 

path = kagglehub.competition_download('ventilator-pressure-prediction')

print("Path to competition files:", path)

df_train = pd.read_csv(path+"/train.csv")
df_test  = pd.read_csv(path+"/test.csv")

100%|██████████| 223M/223M [00:11<00:00, 19.7MB/s] 

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/ventilator-pressure-prediction


#### **Target metric**

The official metric is MAE (mean absolute error). 
    $$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$
It is easy to understand, implement and from what internet says it tends to optimize the model to median solution. But it has it's weaknesses: the error increases linearly, which means that extreme outliers are weighted the same as small errors.

I propose two metrics as an alternative :
1. MSE - it is an industry standart metric that penaltizes outliers rather than small mistakes. It can be used as a loss function for optimization and I will try to train two models for both MAE and MSE and then compare their performance.
    $$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
2. Tolerance accuracy - it is here only because I thought it would be interesting to see what fraction of time our modelpredicts the preassure in some threshold near the actual value, and then I found out there is a metric for that. I will use it mostly to get some insights during validation.
    $$\text{Tolerance Accuracy} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{I}\left(|y_i - \hat{y}_i| \le \epsilon\right)$$

In [4]:
def masked_mae(y_true, y_pred, mask):
    return (abs(y_true - y_pred) * mask).sum() / mask.sum()

def masked_mse(y_true, y_pred, mask):
    return (((y_true - y_pred)**2) * mask).sum() / mask.sum()

def masked_tol(y_true, y_pred, mask, tolerance=1):
    return ((abs(y_true - y_pred) <= tolerance) * mask).sum() / mask.sum()

#### **Splitting and Feature engineering**

In [5]:
STEPS = 80
N_LAGS = 5
BASE_FEATURES = ["R", "C", "time_step", "u_in", "u_out"]
LAG_FEATURES = [f"u_in_lag{n}" for n in range(1, N_LAGS+1)] 
FEATURES = BASE_FEATURES + LAG_FEATURES + ["u_in_cumsum"]
TARGET = "pressure"

for d in (df_train, df_test):
    d.sort_values(["breath_id", "time_step"], inplace=True)
    d.reset_index(drop=True, inplace=True)
assert (df_train.groupby("breath_id").size() == STEPS).all()
assert (df_test.groupby("breath_id").size() == STEPS).all()

In [6]:
breath_groups = df_train[["breath_id", "R", "C"]].drop_duplicates()
breath_groups["R_C"] = breath_groups["R"].astype(str)+'_'+breath_groups["C"].astype(str)

breath_train, breath_val = train_test_split(breath_groups["breath_id"], 
                                            train_size=0.67, 
                                            stratify=breath_groups["R_C"],
                                            random_state=SEED
                                            )
df_val   = df_train[df_train["breath_id"].isin(breath_val)].copy()
df_train = df_train[df_train["breath_id"].isin(breath_train)].copy()

mask_val = (df_val["u_out"] == 0).to_numpy()

In [7]:
def add_lag_cumsum_features(df):
    grouped_df = df.groupby("breath_id")["u_in"]
    for n in range(1, N_LAGS+1):
        df[f"u_in_lag{n}"] = grouped_df.shift(n).fillna(0)
    df["u_in_cumsum"] = grouped_df.cumsum()
    return df

for df in [df_train, df_val, df_test]:
    df = add_lag_cumsum_features(df)

# **ML model**

In [8]:
# DATAPREP

X_train, y_train = df_train[FEATURES].to_numpy(), df_train[TARGET].to_numpy()
X_val, y_val     = df_val[FEATURES].to_numpy(), df_val[TARGET].to_numpy()
X_test = df_test[FEATURES].to_numpy()

In [11]:
# MODELING

wandb.login()
run = wandb.init(project="google_brain", name="ml_cumsum_lag",
                 config=dict(n_estimators=25, 
                             learning_rate=0.1, 
                             max_depth=4))

ml_model = GradientBoostingRegressor(random_state=SEED, **dict(run.config))
ml_model.fit(X_train, y_train)
print("fitted")

fitted


#### **ML Results**

In [12]:
ml_val_pred = ml_model.predict(X_val)

for i, pred in enumerate(ml_model.staged_predict(X_val)):
    wandb.log({"train_loss": ml_model.train_score_[i],
               "val_mae_masked": masked_mae(y_val, pred, mask_val),
               "val_mae_all": mean_absolute_error(y_val, pred)}, step=i)

table = wandb.Table(data=list(zip(FEATURES, ml_model.feature_importances_)),
                    columns=["feature", "importance"])
wandb.log({"feature_importance": wandb.plot.bar(table, "feature", "importance")})

run.summary["val_mae"] = masked_mae(y_val, ml_val_pred, mask_val)
run.summary["val_mse"] = masked_mse(y_val, ml_val_pred, mask_val) 
run.summary["val_tol"] = masked_tol(y_val, ml_val_pred, mask_val, 1)

print(f"MeanAE: {run.summary["val_mae"]}")
print(f"MeanSE: {run.summary["val_mse"]}")
print(f"TolAcc: {run.summary["val_tol"]}")

wandb.finish()

MeanAE: 2.6960040036324435
MeanSE: 14.238800404374848
TolAcc: 0.2705332798876052


train_loss,█▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_mae_all,█▇▆▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
val_mae_masked,█▇▆▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train_loss,6.6292
val_mae,2.696
val_mae_all,1.51546
val_mae_masked,2.696
val_mse,14.2388
val_tol,0.27053


In [13]:
ml_test_pred = ml_model.predict(X_test)

submission = pd.DataFrame({
    'id': df_test['id'],
    'pressure': ml_test_pred.ravel()
})
submission.to_csv('ml_submission.csv', index=False)

print(f"Shape: {submission.shape}")
print(submission.head())

Shape: (4024000, 2)
   id   pressure
0   1   6.650928
1   2   6.510541
2   3   6.751406
3   4   8.998595
4   5  10.314456


In [14]:
# !kaggle competitions submit -c ventilator-pressure-prediction -f ml_submission.csv -m "ml_cumsum_lag"

100% 99.6M/99.6M [00:07<00:00, 14.8MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction

# **DL model**

In [15]:
# DATA PREP

class BreathDataset(Dataset):
    def __init__(self, df, features=FEATURES, target=None, steps=STEPS):
        self.X = torch.tensor(df[features].to_numpy(), dtype=torch.float32).reshape(-1, steps, len(features))
        self.y = None if target is None else torch.tensor(df[target].to_numpy(), dtype=torch.float32).reshape(-1, steps)
        self.mask = torch.tensor((df["u_out"] == 0).to_numpy()).reshape(-1, steps)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx], self.mask[idx]
        else:
            return self.X[idx], self.y[idx], self.mask[idx]
        
train_dataset = BreathDataset(df_train, target=TARGET)
val_dataset   = BreathDataset(df_val,   target=TARGET)
test_dataset  = BreathDataset(df_test)

train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=128)

In [16]:
# CUSTOM LAYER AND OPTIM

class MyLittleLayerFriendshipIsMagic(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        he_init_std = (2/(in_features))**0.5
        self.weight = nn.Parameter(nn.init.normal_(torch.empty(out_features, in_features),
                                                    mean = 0,
                                                    std = he_init_std))
        self.bias   = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return torch.matmul(x, self.weight.T) + self.bias


class TheAmazingWorldOfOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None             
        if closure is not None:
            with torch.enable_grad(): #ya hz what this is for, something to do with lightning
                loss = closure()


        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["beta1"], group["beta2"]
            weight_decay = group["weight_decay"]
            eps = group["eps"]

            for param in group["params"]:
                if param.grad is None:
                    continue
                grad = param.grad

                param.mul_(1 - lr * weight_decay) # AdamW stuff

                state = self.state[param]
                if len(state) == 0:
                    state["step"]       = 0
                    state["exp_avg"]    = torch.zeros_like(param) # m from Adam formula
                    state["exp_avg_sq"] = torch.zeros_like(param) # v
                exp_avg    = state["exp_avg"] 
                exp_avg_sq = state["exp_avg_sq"]
                state["step"] += 1
                t = state["step"]

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)  # розібратись with this shi
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1 - beta1 ** t
                bias_correction2 = 1 - beta2 ** t

                denom = exp_avg_sq.sqrt() / bias_correction2 ** 0.5 + eps
                param.addcdiv_(exp_avg, denom, value=-lr / bias_correction1)
        
        return loss

In [17]:
# MODELING

class PressureMLP(nn.Module):
    def __init__(self, in_features):
        super(PressureMLP, self).__init__()
        self.input_norm = nn.BatchNorm1d(in_features)
        self.layers = nn.Sequential(
            nn.Linear(in_features, 16),
            nn.ReLU(),
            nn.Linear(16, 32), # Here was my little layer
            nn.ReLU(),
            nn.Linear(32, 64), # Here was my little layer
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        x = self.input_norm(x.transpose(1, 2)).transpose(1, 2)
        return self.layers(x)

In [18]:
class LitPressureMLP(pl.LightningModule):
    def __init__(self, in_features, lr=1e-3, loss_fn=nn.L1Loss(reduction="none")):
        super().__init__()
        self.model = PressureMLP(in_features)
        self.loss_fn = loss_fn
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y, mask = batch
        preds = self(x).squeeze(-1)
        loss = (self.loss_fn(preds, y) * mask).sum() / mask.sum()
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, mask = batch
        preds = self(x).squeeze(-1)
        
        self.log_dict({"val_mae":masked_mae(y, preds, mask), 
                       "val_mse":masked_mse(y, preds, mask), 
                       "val_tol":masked_tol(y, preds, mask, 1)}, 
                        prog_bar=True, batch_size=int(mask.sum()))

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr) # TheAmazingWorldOfOptimizer

dl_model = LitPressureMLP(in_features=len(FEATURES))

In [19]:
wandb.login()
wandb_logger = WandbLogger(project="google_brain", name="mae_batchnorm_model")
wandb_logger.watch(dl_model, log='gradients', log_freq=50)

trainer = pl.Trainer(
    max_epochs=16,
    val_check_interval=0.5,
    log_every_n_steps=10,
    accelerator="auto",
    logger=wandb_logger,
)
trainer.fit(model=dl_model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ PressureMLP │ 15.4 K │ train │     0 │
│ 1 │ loss_fn │ L1Loss      │      0 │ train │     0 │
└───┴─────────┴─────────────┴────────┴───────┴───────┘

Trainable params: 15.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 15.4 K                                                                                               
Total estimated model params size (MB): 0.061                                                                      
Modules in train mode: 15                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=16` reached.


epoch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
train_loss,█▆▅▄▄▃▂▃▂▅▂▂▁▂▃▂▂▂▂▃▁▁▄▄▂▂▁▃▂▁▂▄▂▄▄▂▁▂▁▁
trainer/global_step,▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇█
val_mae,█▅▄▄▃▃▃▄▂▂▂▂▂▂▁▁▁▂▂▁▁▂▃▂▂▂▂▁▁▁▃▁
val_mse,█▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▂▂▁▁▂▂▁▂▁▂▁▁▁▂▁
val_tol,▁▃▄▅▅▆▅▄▆▆▆▇▇▇▇██▅███▆▆▆▇▆▅███▄█
epoch,15
train_loss,2.0992
trainer/global_step,6319
val_mae,1.64632
val_mse,5.6687


#### **DL Resulsts**

In [20]:
dl_model.eval()
with torch.no_grad():
    dl_val_pred  = dl_model(val_dataset.X.to(dl_model.device)).squeeze(-1).reshape(-1).cpu().numpy()
    dl_test_pred = dl_model(test_dataset.X.to(dl_model.device)).squeeze(-1).reshape(-1).cpu().numpy()

print(f"MeanAE: {masked_mae(y_val, dl_val_pred, mask_val)}")
print(f"MeanSE: {masked_mse(y_val, dl_val_pred, mask_val)}")
print(f"TolAcc: {masked_tol(y_val, dl_val_pred, mask_val, 1)}")

pd.DataFrame({'id': df_test['id'], 'pressure': dl_test_pred}).to_csv('dl_submission.csv', index=False)

MeanAE: 1.6829013116953193
MeanSE: 5.79937754216132
TolAcc: 0.45906837745913187


In [21]:
# !kaggle competitions submit -c ventilator-pressure-prediction -f dl_submission.csv -m "lag_features_cumsum_batchnorm_custom_adam_dl"

100% 66.8M/66.8M [00:01<00:00, 46.1MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction

# **Ensamble**

In [21]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor

def to_ensamble_data(df, ml_pred, dl_pred):
    return pd.DataFrame({"ml": ml_pred, "dl": dl_pred,
                         "R": df["R"].to_numpy(), "C": df["C"].to_numpy(),
                         "u_out": df["u_out"].to_numpy(), "time_step": df["time_step"].to_numpy()})

meta_val  = to_ensamble_data(df_val,  ml_val_pred,  dl_val_pred)
meta_test = to_ensamble_data(df_test, ml_test_pred, dl_test_pred)
groups    = df_val["breath_id"].to_numpy()

In [22]:
mean_pred = (ml_val_pred + dl_val_pred) / 2
ws = np.linspace(0, 1, 21)
w_scores = [masked_mae(y_val, w * dl_val_pred + (1 - w) * ml_val_pred, mask_val) for w in ws]
best_w = ws[int(np.argmin(w_scores))]
weighted_pred = best_w * dl_val_pred + (1 - best_w) * ml_val_pred
print(f"best weight for weighted average: {best_w:.2f}")

best weight for weighted average: 0.95


In [23]:
oof = np.zeros(len(meta_val))
for fold, (tr_idx, te_idx) in enumerate(GroupKFold(n_splits=5).split(meta_val, y_val, groups=groups)):
    fold_model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=100, random_state=SEED)
    stacker = fold_model.fit(meta_val.iloc[tr_idx], y_val[tr_idx])
    oof[te_idx] = stacker.predict(meta_val.iloc[te_idx])
    print(f"fold {fold}: masked MAE = {masked_mae(y_val[te_idx], oof[te_idx], mask_val[te_idx]):.4f}")

results = pd.DataFrame({
    "model": ["ML", "DL", "mean", f"weighted (w={best_w:.2f})", "stacking (OOF)"],
    "masked_mae": [masked_mae(y_val, p, mask_val) for p in
                   (ml_val_pred, dl_val_pred, mean_pred, weighted_pred, oof)],
    "masked_mse": [masked_mse(y_val, p, mask_val) for p in
                   (ml_val_pred, dl_val_pred, mean_pred, weighted_pred, oof)],
    "tol@1":      [masked_tol(y_val, p, mask_val, 1) for p in
                   (ml_val_pred, dl_val_pred, mean_pred, weighted_pred, oof)],
})
print(results.to_string(index=False))

fold 0: masked MAE = 1.6169
fold 1: masked MAE = 1.6318
fold 2: masked MAE = 1.6159
fold 3: masked MAE = 1.6043
fold 4: masked MAE = 1.5625
            model  masked_mae  masked_mse    tol@1
               ML    2.696004   14.238800 0.270533
               DL    1.682901    5.799378 0.459068
             mean    1.972913    7.583624 0.373666
weighted (w=0.95)    1.680352    5.758610 0.457009
   stacking (OOF)    1.606304    5.589323 0.490949


In [25]:
ens_model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=100, random_state=SEED)
ens_model.fit(meta_val, y_val)         # all validation breaths
ensamble_test_pred = ens_model.predict(meta_test)

pd.DataFrame({"id": df_test["id"], "pressure": ensamble_test_pred}).to_csv("ensamble_submission.csv", index=False)
print(pd.read_csv("ensamble_submission.csv").shape)             # expect (4024000, 2)

(4024000, 2)


In [26]:
!kaggle competitions submit -c ventilator-pressure-prediction -f ensamble_submission.csv -m "stacking HistGradBoost"

100% 99.6M/99.6M [00:06<00:00, 15.1MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction